In [1]:
import pandas as pd
df=pd.read_json('regex_output.ndjson',lines=True)

In [2]:
df_ml = df[df["classified_by"] == "Unclassified"].copy()

In [3]:
SEMANTIC_SEEDS = {
    "SECURITY_EVENT": [
        "permission denied", "unauthorized", "authentication failed",
        "access denied", "forbidden"
    ],
    "RESOURCE_PRESSURE": [
        "memory pressure", "cpu pressure", "thermal",
        "out of memory", "low memory"
    ],
    "RESOURCE_FAILURE": [
        "failed to allocate", "cannot allocate",
        "no space left", "disk full"
    ],
    "HARDWARE_EVENT": [
        "usb", "thunderbolt", "pci", "wifi", "bluetooth"
    ],
    "LIFECYCLE_EVENT": [
        "started", "stopped", "exited", "shutdown", "initialized"
    ],
}


In [4]:
def assign_semantic_type(text):
    text = text.lower()
    for semantic, keywords in SEMANTIC_SEEDS.items():
        for kw in keywords:
            if kw in text:
                return semantic
    return None

In [5]:
df_ml["semantic_type"] = df_ml["clean_message"].apply(assign_semantic_type)

In [6]:
df_ml["semantic_type"].value_counts(dropna=False)

,count
semantic_type,
None,8326
HARDWARE_EVENT,143
SECURITY_EVENT,98
LIFECYCLE_EVENT,64
RESOURCE_PRESSURE,39


In [7]:
df_train = df_ml[df_ml["semantic_type"].notna()].copy()

print(df_train["semantic_type"].value_counts())



semantic_type
HARDWARE_EVENT       143
SECURITY_EVENT        98
LIFECYCLE_EVENT       64
RESOURCE_PRESSURE     39
Name: count, dtype: int64


In [8]:
texts = df_train["clean_message"].tolist()
labels = df_train["semantic_type"].tolist()

In [9]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)

Classes: ['HARDWARE_EVENT' 'LIFECYCLE_EVENT' 'RESOURCE_PRESSURE' 'SECURITY_EVENT']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_enc = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=64
)

val_enc = tokenizer(
    X_val,
    truncation=True,
    padding=True,
    max_length=64
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:
import torch

class LogDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = LogDataset(train_enc, y_train)
val_ds   = LogDataset(val_enc, y_val)


In [13]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


In [14]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

# Override loss to include class weights
def weighted_loss(outputs, labels):
    loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.device))
    return loss_fct(outputs.logits, labels)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )
        loss = loss_fct(outputs.logits, labels)

        return (loss, outputs) if return_outputs else loss


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./semantic_model_step7",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=12,
    weight_decay=0.01,
    logging_steps=20,
    save_steps=500,
    eval_steps=500,
    save_total_limit=2,
    report_to=None
)


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

trainer.train()


/tmp/ipython-input-2528607371.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,1.212500
40,0.675800
60,0.310000


In [ ]:
model.save_pretrained("./semantic_model_step7")
tokenizer.save_pretrained("./semantic_model_step7")

import joblib
joblib.dump(label_encoder, "./semantic_model_step7/label_encoder.pkl")


In [ ]:
import torch
import joblib
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import torch.nn.functional as F

model = DistilBertForSequenceClassification.from_pretrained(
    "./semantic_model_step7"
)
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "./semantic_model_step7"
)
label_encoder = joblib.load(
    "./semantic_model_step7/label_encoder.pkl"
)

model.eval()


In [ ]:
def predict_semantic(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

    return (
        label_encoder.inverse_transform([pred.item()])[0],
        conf.item()
    )


In [ ]:
df_unlabeled = df_ml[df_ml["semantic_type"].isna()].copy()

CONF_THRESHOLD = 0.85

new_labels = []
new_confs = []

for text in df_unlabeled["clean_message"]:
    sem, conf = predict_semantic(text)
    if conf >= CONF_THRESHOLD:
        new_labels.append(sem)
        new_confs.append(conf)
    else:
        new_labels.append(None)
        new_confs.append(conf)

df_unlabeled["semantic_type_pred"] = new_labels
df_unlabeled["semantic_confidence"] = new_confs


In [ ]:
df_accepted = df_unlabeled[
    df_unlabeled["semantic_type_pred"].notna()
].copy()

print(df_accepted["semantic_type_pred"].value_counts())


In [ ]:
df_final = pd.concat(
    [
        df_train[["clean_message", "semantic_type"]].reset_index(drop=True),
        df_accepted[["clean_message", "semantic_type"]].reset_index(drop=True)
    ],
    ignore_index=True
)

print(df_final["semantic_type"].value_counts())
print("Final training size:", len(df_final))


In [ ]:
print("df_train columns:")
print(df_train.columns.tolist())

print("\ndf_accepted columns:")
print(df_accepted.columns.tolist())


In [ ]:
# Remove duplicate columns, keep last occurrence
df_accepted = df_accepted.loc[:, ~df_accepted.columns.duplicated(keep="last")]
df_accepted.columns.tolist()


In [ ]:
df_train_clean = df_train[["clean_message", "semantic_type"]].copy()
df_accepted_clean = df_accepted[["clean_message", "semantic_type"]].copy()

df_final = pd.concat(
    [df_train_clean, df_accepted_clean],
    ignore_index=True
)

print(df_final["semantic_type"].value_counts())
print("Final training size:", len(df_final))


In [ ]:
df_train_clean = df_train[["clean_message", "semantic_type"]].copy()
df_accepted_clean = df_accepted[["clean_message", "semantic_type"]].copy()

df_final = pd.concat(
    [df_train_clean, df_accepted_clean],
    ignore_index=True
)

print(df_final["semantic_type"].value_counts())
print("Final training size:", len(df_final))


In [ ]:
from sklearn.preprocessing import LabelEncoder

texts = df_final["clean_message"].tolist()
labels = df_final["semantic_type"].tolist()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

num_labels = len(label_encoder.classes_)
print("Classes:", label_encoder.classes_)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)


In [ ]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

train_enc = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=64
)

val_enc = tokenizer(
    X_val,
    truncation=True,
    padding=True,
    max_length=64
)


In [ ]:
import torch

class LogDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = LogDataset(train_enc, y_train)
val_ds = LogDataset(val_enc, y_val)


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float)


In [33]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [34]:
from transformers import TrainingArguments
import os

os.environ["WANDB_DISABLED"] = "true"

training_args = TrainingArguments(
    output_dir="./semantic_model_final",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,   # FINAL
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    save_total_limit=2,
    report_to=None
)


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [38]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer  # warning is OK
)

trainer.train()


/tmp/ipython-input-2978107013.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedTrainer.__init__`. Use `processing_class` instead.
  trainer = WeightedTrainer(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.219000
100,0.155900
150,0.163500
200,0.116200
250,0.167500
300,0.148100
350,0.161800
400,0.006400
450,0.117600
500,0.006200


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1316, training_loss=0.058716765344482606, metrics={'train_runtime': 4880.0708, 'train_samples_per_second': 4.311, 'train_steps_per_second': 0.27, 'total_flos': 348407897548800.0, 'train_loss': 0.058716765344482606, 'epoch': 4.0})

In [39]:
model.save_pretrained("./semantic_model_final")
tokenizer.save_pretrained("./semantic_model_final")

import joblib
joblib.dump(label_encoder, "./semantic_model_final/label_encoder.pkl")


['./semantic_model_final/label_encoder.pkl']

In [40]:
SEVERITY_MAP = {
    # 🚨 Always serious
    "SECURITY_EVENT": "ERROR",
    "RESOURCE_FAILURE": "ERROR",

    # ⚠️ Potentially serious
    "RESOURCE_PRESSURE": "WARN",
    "TRANSIENT_FAILURE": "WARN",

    # ℹ️ Informational noise
    "HARDWARE_EVENT": "INFO",
    "LIFECYCLE_EVENT": "INFO",
}


In [45]:
import torch
import joblib
import torch.nn.functional as F
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast

MODEL_DIR = "./semantic_model_final"

model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
label_encoder = joblib.load(f"{MODEL_DIR}/label_encoder.pkl")

model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [49]:
def predict_semantic_type(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    conf, pred = torch.max(probs, dim=1)

    semantic_type = label_encoder.inverse_transform([pred.item()])[0]

    return {
        "semantic_type": semantic_type,
        "confidence": float(conf.item())
    }


In [63]:
CONFIDENCE_THRESHOLD = 0.75

In [53]:
def classify_log_after_regex(clean_message: str):
    semantic_result = predict_semantic_type(clean_message)

    semantic_type = semantic_result["semantic_type"]
    confidence = semantic_result["confidence"]

    # Low confidence → escalate
    if confidence < CONFIDENCE_THRESHOLD or semantic_type is None:
        return {
            "clean_message": clean_message,
            "semantic_type": semantic_type,
            "confidence": confidence,
            "severity": "UNKNOWN",
            "action": "ESCALATE_TO_LLM"
        }

    severity = resolve_severity(semantic_type)

    return {
        "clean_message": clean_message,
        "semantic_type": semantic_type,
        "confidence": confidence,
        "severity": severity,
        "action": "AUTO_CLASSIFIED"
    }


In [54]:
log = "permission denied while accessing config file"
result = classify_log_after_regex(log)
print(result)


{'clean_message': 'permission denied while accessing config file', 'semantic_type': 'SECURITY_EVENT', 'confidence': 0.9978669881820679, 'severity': 'ERROR', 'action': 'AUTO_CLASSIFIED'}


In [55]:
log = "process exited normally with status 0"

classify_log_after_regex(log)


{'clean_message': 'process exited normally with status 0',
 'semantic_type': None,
 'confidence': 0.9764277338981628,
 'severity': 'UNKNOWN',
 'action': 'ESCALATE_TO_LLM'}

In [65]:
log = "unexpected packet sequence observed in module xyz"

classify_log_after_regex(log)


{'clean_message': 'unexpected packet sequence observed in module xyz',
 'semantic_type': None,
 'confidence': 0.9996034502983093,
 'severity': 'UNKNOWN',
 'action': 'ESCALATE_TO_LLM'}

In [66]:
log = "failed to connect to database"
classify_log_after_regex(log)

{'clean_message': 'failed to connect to database',
 'semantic_type': None,
 'confidence': 0.9985333681106567,
 'severity': 'UNKNOWN',
 'action': 'ESCALATE_TO_LLM'}

In [ ]:
ALLOWED_SEMANTIC_TYPES = [
    "SECURITY_EVENT",
    "RESOURCE_PRESSURE",
    "RESOURCE_FAILURE",
    "TRANSIENT_FAILURE",
    "LIFECYCLE_EVENT",
    "HARDWARE_EVENT",
    "UNKNOWN"
]
ALLOWED_SEVERITY = {"INFO", "WARN", "ERROR"}

In [ ]:
LLM_PROMPT_TEMPLATE = """
You are a log analysis expert.

Classify the following log message into EXACTLY ONE semantic type from the list.
Do NOT invent new categories.
Do NOT add explanations outside JSON.

Semantic types:
- SECURITY_EVENT
- RESOURCE_PRESSURE
- RESOURCE_FAILURE
- TRANSIENT_FAILURE
- LIFECYCLE_EVENT
- HARDWARE_EVENT
- UNKNOWN

Log message:
"{log_message}"

Respond ONLY with a SINGLE valid JSON object in this exact format:

{{
  "semantic_type": "SECURITY_EVENT | RESOURCE_PRESSURE | RESOURCE_FAILURE | TRANSIENT_FAILURE | LIFECYCLE_EVENT | HARDWARE_EVENT | UNKNOWN",
  "severity": "INFO | WARN | ERROR",
  "reason": "short reason (max 20 words)"
}}
"""


In [79]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def call_llm(prompt: str) -> str | None:
    try:
        chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile",
            temperature=0.2,   # LOWER = more structured
            max_tokens=512,
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        print("Groq API Error:", e)
        return None


In [ ]:
import json
import re

def safe_parse_llm_response(raw_response: str | None):
    if raw_response is None:
        return {
            "semantic_type": "UNKNOWN",
            "severity": "UNKNOWN",
            "reason": "LLM call failed"
        }

    # 🔑 Extract JSON block using regex
    match = re.search(r"\{.*\}", raw_response, re.DOTALL)
    if not match:
        return {
            "semantic_type": "UNKNOWN",
            "severity": "UNKNOWN",
            "reason": "No JSON found in LLM response"
        }

    json_text = match.group(0)

    try:
        data = json.loads(json_text)
    except Exception:
        return {
            "semantic_type": "UNKNOWN",
            "severity": "UNKNOWN",
            "reason": "Malformed JSON from LLM"
        }

    semantic_type = data.get("semantic_type", "UNKNOWN")
    severity = data.get("severity", "UNKNOWN")
    reason = data.get("reason", "")

    if semantic_type not in ALLOWED_SEMANTIC_TYPES:
        semantic_type = "UNKNOWN"

    if severity not in ALLOWED_SEVERITY:
        severity = "UNKNOWN"

    return {
        "semantic_type": semantic_type,
        "severity": severity,
        "reason": reason[:200]
    }

In [72]:
def llm_fallback(clean_message: str):
    prompt = LLM_PROMPT_TEMPLATE.format(log_message=clean_message)

    raw_response = call_llm(prompt)

    result = safe_parse_llm_response(raw_response)

    return {
        "clean_message": clean_message,
        "semantic_type": result["semantic_type"],
        "severity": result["severity"],
        "reason": result["reason"],
        "source": "LLM"
    }


In [88]:
log = "temporary failure in name resolution, retrying"
llm_fallback(log)

KeyError: '\n  "semantic_type"'

In [93]:
log = "temporary failure in name resolution, retrying"
print(llm_fallback(log))


{'clean_message': 'temporary failure in name resolution, retrying', 'semantic_type': 'TRANSIENT_FAILURE', 'severity': 'WARN', 'reason': 'Temporary name resolution failure', 'source': 'LLM'}
